In [1]:
%uv pip install torch

Using Python 3.12.6 environment at: /usr/local
Audited 1 package in 26ms
Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch

In [3]:
def rms_norm_ref(x, weight, eps=1e-6):
    x_fp32 = x.float()
    mean_sqr = x_fp32.square().mean(dim=-1, keepdim=True)
    reciprocal_rms = torch.rsqrt(mean_sqr + eps)

    weight_fp32 = weight.float()
    output_fp32 = x_fp32 * reciprocal_rms * weight_fp32

    return output_fp32.to(dtype=x.dtype)
    

In [4]:
x = torch.tensor([
    [1.0, 2.0, 3.0],
    [0.0, 0.0, 0.0]
])

weight = torch.ones(3)

output = rms_norm_ref(x, weight)

print(output)
print(output.square().mean(dim=-1))

tensor([[0.4629, 0.9258, 1.3887],
        [0.0000, 0.0000, 0.0000]])
tensor([1.0000, 0.0000])


In [5]:
expected = torch.nn.functional.rms_norm(
    x, normalized_shape=(x.shape[-1],),
    weight=weight,
    eps=1e-6
)

torch.testing.assert_close(output, expected)

print("reference matches torch")
print("max error:", (output-expected).abs().max().item())

reference matches torch
max error: 0.0


In [6]:
torch.manual_seed(0)

x = torch.randn(2, 3, 33)
weight = torch.randn(33)

actual = rms_norm_ref(x, weight, eps=1e-6)

expected = torch.nn.functional.rms_norm(
    x,
    normalized_shape=(33,),
    weight=weight,
    eps=1e-6
)

torch.testing.assert_close(actual, expected)

print("shape:", actual.shape)
print("dtype:", actual.dtype)
print("max error:", (actual-expected).abs().max().item())

shape: torch.Size([2, 3, 33])
dtype: torch.float32
max error: 0.0


In [8]:
x_fp16 = torch.full((2, 33), 300.0, dtype=torch.float16)
weight_fp16 = torch.ones(33, dtype=torch.float16)

unsafe_ms = x_fp16.square().mean(dim=-1)
safe_ms = x_fp16.float().square().mean(dim=-1)

output = rms_norm_ref(x_fp16, weight_fp16)

print("FP16 mean square:", unsafe_ms)
print("FP32 mean square:", safe_ms)
print("output finite:", torch.isfinite(output).all().item())
print("first output value:", output[0, 0].item())

FP16 mean square: tensor([inf, inf], dtype=torch.float16)
FP32 mean square: tensor([90000., 90000.])
output finite: True
first output value: 1.0
